# 1. Objective

Reproduce the classification component of **“Weakly Supervised Pneumonia Localization”** with a custom 10-convolution-layer CNN. The network classifies an X-ray as Healthy (`Normal`, label 0) or Pneumonia (`Lung Opacity`, label 1). It ends in global average pooling and one linear classifier so the final convolution maps and classifier weights remain available for a later CAM stage. This notebook does not implement CAM.

# 2. Dataset

This experiment reuses the permanent split from `metadata/split_metadata.csv`; it does not regenerate or alter the split. The local usable cohort has 14,863 labeled images, which differs from the paper's reported cohort size.

In [ ]:
import csv
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "metadata").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "metadata").is_dir():
    raise FileNotFoundError("Could not locate the project root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

with (ROOT / "metadata" / "split_metadata.csv").open(newline="", encoding="utf-8-sig") as f:
    split_rows = list(csv.DictReader(f))
summary = {}
for split in ("train", "validation", "test"):
    rows = [row for row in split_rows if row["split"] == split]
    summary[split] = {
        "total": len(rows),
        "healthy": sum(row["label"] == "0" for row in rows),
        "pneumonia": sum(row["label"] == "1" for row in rows),
    }
display(Markdown("| Split | Total | Healthy | Pneumonia |\n|---|---:|---:|---:|\n" + "\n".join(
    f"| {split} | {values['total']:,} | {values['healthy']:,} | {values['pneumonia']:,} |"
    for split, values in summary.items()
)))
print("Image dimensions: 128 × 128 grayscale")
print("Total labeled images:", len(split_rows))

# 3. Preprocessing

The shared dataset loader decodes original DICOM files on demand, resizes each grayscale image to **128 × 128**, scales pixels to [0, 1], and standardizes with the fixed global mean and standard deviation fitted on the training split only. No full resized image copy is created. Training consumes a one-channel tensor; labels and box metadata continue to come from the same split row.

In [ ]:
preprocessing = json.loads((ROOT / "metadata" / "preprocessing_stats.json").read_text(encoding="utf-8"))
print(f"Resize: {preprocessing['image_size'][0]} × {preprocessing['image_size'][1]}")
print(f"Training-set mean: {preprocessing['mean']:.8f}")
print(f"Training-set standard deviation: {preprocessing['std']:.8f}")
print(f"Statistics fitted on {preprocessing['fit_image_count']:,} training images")

# 4. CNN Architecture

The paper description does not provide channel widths or exact pooling locations. This explicit implementation uses five two-convolution blocks with widths **32, 64, 96, 128, 128**, each convolution using a 3 × 3 kernel, same padding, and ReLU; 2 × 2 max pooling follows the first four blocks. This preserves spatial shape inside each block and produces 8 × 8 final convolution maps. Global average pooling reduces each channel to one value, followed by exactly one two-class fully connected layer. The final maps are returned by `forward_features`; classifier weights are available as `model.classifier.weight` for the future CAM stage.

In [ ]:
import torch
from src.models.cnn import Custom10LayerCNN, count_parameters

model = Custom10LayerCNN()
model.summary()
cam_check = model.verify_cam_compatibility()
print("CAM compatibility structure check:", cam_check)
print("Architecture:", model.architecture_config())

# 5. Training Configuration

- Optimizer: **Adam**
- Learning rate: **0.0001**
- Epochs: **20**
- Seed: **42**
- Batch size: **32**
- Device: CUDA or supported Intel XPU when available; otherwise CPU
- Best checkpoint: highest validation accuracy (validation loss breaks ties)

Training uses only the train split. Validation is used for checkpoint selection. The test split is loaded only after the best checkpoint is restored.

# 6. Training

Run the cell below to perform the full 20-epoch training and write the best checkpoint and `results/cnn_results.json`. To inspect an existing run without retraining, set `RUN_TRAINING = False`; the curves are loaded from the saved epoch history.

In [ ]:
from scripts.train_cnn import run_training

RUN_TRAINING = True
if RUN_TRAINING:
    cnn_result = run_training()
else:
    cnn_result = json.loads((ROOT / "results" / "cnn_results.json").read_text(encoding="utf-8"))
history = cnn_result["history"]
print(f"Best epoch: {cnn_result['best_epoch']}")
print(f"Train accuracy: {cnn_result['train_accuracy']:.2%}")
print(f"Best-checkpoint validation accuracy: {cnn_result['validation_accuracy']:.2%}")

In [ ]:
from PIL import Image, ImageDraw

def draw_curves(history, first_key, second_key, title, output_path):
    width, height = 760, 340
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    left, right, top, bottom = 70, 730, 50, 285
    draw.text((left, 18), title, fill="#222222")
    for step in range(6):
        y = bottom - step * (bottom - top) / 5
        draw.line((left, y, right, y), fill="#e6e6e6", width=1)
    maximum_epoch = max(item["epoch"] for item in history)
    values = [item[first_key] for item in history] + [item[second_key] for item in history]
    low, high = min(values), max(values)
    if high == low:
        high = low + 1.0
    def points(key):
        return [
            (left + (right-left) * (item["epoch"]-1) / max(maximum_epoch-1, 1),
             bottom - (bottom-top) * (item[key]-low) / (high-low))
            for item in history
        ]
    draw.line(points(first_key), fill="#2b6cb0", width=3)
    draw.line(points(second_key), fill="#dd6b20", width=3)
    draw.text((left, bottom+12), f"range: {low:.3f} to {high:.3f}", fill="#333333")
    draw.line((left+300, 25, left+330, 25), fill="#2b6cb0", width=3)
    draw.text((left+338, 18), first_key.replace("_", " "), fill="#333333")
    draw.line((left+500, 25, left+530, 25), fill="#dd6b20", width=3)
    draw.text((left+538, 18), second_key.replace("_", " "), fill="#333333")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path)
    return image

loss_chart = draw_curves(history, "training_loss", "validation_loss", "Loss by epoch", ROOT / "results" / "cnn_loss_curve.png")
accuracy_chart = draw_curves(history, "training_accuracy", "validation_accuracy", "Accuracy by epoch", ROOT / "results" / "cnn_accuracy_curve.png")
display(loss_chart, accuracy_chart)

# 7. Evaluation

The training script restores the checkpoint selected on validation accuracy before evaluating the test split. Sensitivity treats Pneumonia as positive (label 1); specificity measures Healthy/Normal recognition (label 0).

In [ ]:
metrics = {
    "Train accuracy": cnn_result["train_accuracy"],
    "Validation accuracy": cnn_result["validation_accuracy"],
    "Test accuracy": cnn_result["test_accuracy"],
    "Sensitivity": cnn_result["sensitivity"],
    "Specificity": cnn_result["specificity"],
}
display(Markdown("| Metric | Result |\n|---|---:|\n" + "\n".join(f"| {name} | {value:.2%} |" for name, value in metrics.items())))
print("Confusion matrix (rows=true Healthy/Pneumonia, columns=predicted Healthy/Pneumonia):")
print(cnn_result["confusion_matrix"])

# 8. Reference Comparison

Both measured values below are loaded from the actual trained run's JSON result; reference values are stored separately as paper-reported targets.

In [ ]:
reference_train = cnn_result["reference"]["train_accuracy"]
reference_test = cnn_result["reference"]["test_accuracy"]
ours_train = cnn_result["train_accuracy"]
ours_test = cnn_result["test_accuracy"]
comparison = (
    "| Metric | Reference | Our Result | Difference |\n|---|---:|---:|---:|\n"
    f"| Train Accuracy | {reference_train:.2%} | {ours_train:.2%} | {(ours_train-reference_train)*100:+.2f} pp |\n"
    f"| Test Accuracy | {reference_test:.2%} | {ours_test:.2%} | {(ours_test-reference_test)*100:+.2f} pp |"
)
display(Markdown(comparison))

# 9. Discussion

Compare the measured result to the reference targets while accounting for the local cohort difference: the project has 14,863 usable labeled images versus 17,489 reported by the paper. The paper summary provided for this implementation also leaves channel widths and some training details unspecified; the explicit channel schedule and pooling choices are therefore documented experiment choices, not asserted paper details. Differences may reflect cohort composition, split, implementation details, and optimization. The final convolution maps and single linear classifier are deliberately exposed for the planned CAM stage, which is not performed here.